In [17]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, gaussian_kde
import os
import seaborn as sns

In [18]:
# ---- Config ----
PICNIC_PATH = "../source_data/PICNIC-9606-data.csv"
PHPHUNTER_PATH = "../source_data/AllhumanReviewed.txt"
MICROEXON_PATH = "../outputs/microexon_final.csv"
OUT_DIR = "../outputs/"
FONT_SIZE = 14  # large, uniform Arial

In [19]:
# Colors (consistent with all previous figures)
C_MICRO = "#E63946"   # coral red — microexon
C_REST = "#4C72B0"    # steel blue — all human
C_MICRO_EDGE = "#A4161A"
C_REST_EDGE = "#2F4B7C"

# ---- Fonts: Arial ----
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE, "axes.titlesize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE, "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "figure.dpi": 300, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.linewidth": 0.6,
    "svg.fonttype": "none",
})

# ---- Load data ----
micro = pd.read_csv(MICROEXON_PATH)
micro_genes = set(micro["GENE"].dropna().unique())
print(f"Microexon genes: {len(micro_genes)}")

picnic = pd.read_csv(PICNIC_PATH)
picnic["gene_list"] = picnic["Genes"].fillna("").str.split()
picnic["is_microexon"] = picnic["gene_list"].apply(
    lambda gl: any(g in micro_genes for g in gl))
print(f"PICNIC: {len(picnic)} rows, {picnic['is_microexon'].sum()} microexon matches")

php = pd.read_csv(PHPHUNTER_PATH, sep="\t")
# Bridge via Uniprot IDs from PICNIC matches
uniprot_micro = set(picnic[picnic["is_microexon"]]["Uniprot ID"].unique())
php["is_microexon"] = php["Gene"].isin(micro_genes) | php["Uniprot"].isin(uniprot_micro)
print(f"PHPhunter: {len(php)} rows, {php['is_microexon'].sum()} microexon matches")

# ---- Statistical tests ----
def cliff_delta(x, y):
    """Cliff's delta: proportion of x > y minus proportion of x < y."""
    x, y = np.asarray(x), np.asarray(y)
    n = len(x) * len(y)
    # Count pairs where x > y and x < y
    greater = sum(np.sum(x_i > y) for x_i in x)
    less = sum(np.sum(x_i < y) for x_i in x)
    return (greater - less) / n

stats = {}
for name, df, col in [("PICNIC", picnic, "PICNIC score"),
                       ("PHPhunter", php, "Probability")]:
    micro_scores = df[df["is_microexon"]][col].dropna().values
    all_scores = df[~df["is_microexon"]][col].dropna().values
    u, p = mannwhitneyu(micro_scores, all_scores, alternative='two-sided')
    delta = cliff_delta(micro_scores, all_scores)
    stats[name] = {
        "micro_n": len(micro_scores), "all_n": len(all_scores),
        "micro_median": np.median(micro_scores), "all_median": np.median(all_scores),
        "micro_mean": np.mean(micro_scores), "all_mean": np.mean(all_scores),
        "U": u, "p": p, "cliff_delta": delta,
    }
    print(f"\n{name}:")
    print(f"  Microexon (n={len(micro_scores)}): median={np.median(micro_scores):.3f}, mean={np.mean(micro_scores):.3f}")
    print(f"  All human (n={len(all_scores)}): median={np.median(all_scores):.3f}, mean={np.mean(all_scores):.3f}")
    print(f"  Mann-Whitney U: U={u:.0f}, p={p:.2e}")
    print(f"  Cliff's delta: {delta:.3f}")

# ---- Figure: 2-panel shaded KDE comparison ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor="white")

for ax, (name, df, col, xlabel) in zip(
    axes,
    [("PICNIC", picnic, "PICNIC score", "PICNIC phase separation score"),
     ("PHPhunter", php, "Probability", "PHPhunter phase separation probability")]):

    micro_scores = df[df["is_microexon"]][col].dropna().values
    all_scores = df[~df["is_microexon"]][col].dropna().values
    s = stats[name]

    # Shaded KDE curves via seaborn
    sns.kdeplot(all_scores, ax=ax, fill=True, alpha=0.35, color=C_REST,
                linewidth=1.8, label=f"All human genes")
    sns.kdeplot(micro_scores, ax=ax, fill=True, alpha=0.45, color=C_MICRO,
                linewidth=1.8, label=f"Microexon genes")

    # p-value annotation (no box, no test name)
    ptxt = f"p = {s['p']:.1e}" if s["p"] < 0.001 else f"p = {s['p']:.3f}"
    ax.text(0.97, 0.97, ptxt,
            transform=ax.transAxes, ha="right", va="top",
            fontsize=FONT_SIZE, color="#333333")

    ax.set_xlim(-0.1, 1.1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(name, fontweight="bold", fontsize=FONT_SIZE)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(loc="upper left", frameon=False, fontsize=FONT_SIZE)

fig.tight_layout()

path_png = os.path.join(OUT_DIR, "microexon_phasesep_distribution.png")
path_svg = os.path.join(OUT_DIR, "microexon_phasesep_distribution.svg")
fig.savefig(path_png, bbox_inches="tight", facecolor="white")
fig.savefig(path_svg, bbox_inches="tight", facecolor="white")
plt.close()
print(f"\nSaved: {path_png}")
print(f"Saved: {path_svg}")

# ---- Save per-gene scores table ----
# Build merged table: gene, uniprot, PICNIC score, PHPhunter probability, is_microexon
picnic_out = picnic[["Uniprot ID", "Genes", "PICNIC score", "is_microexon"]].copy()
picnic_out.columns = ["Uniprot", "Gene", "PICNIC_score", "is_microexon"]
# Take first gene name from space-separated list
picnic_out["Gene"] = picnic_out["Gene"].fillna("").str.split().str[0]

php_out = php[["Uniprot", "Gene", "Probability", "is_microexon"]].copy()
php_out.columns = ["Uniprot", "Gene", "PHPhunter_probability", "is_microexon"]

# Merge on Uniprot
merged = picnic_out.merge(php_out[["Uniprot", "PHPhunter_probability"]],
                          on="Uniprot", how="outer")
# Use microexon flag from either source
merged["is_microexon"] = merged["is_microexon"].fillna(False) | \
    merged["Uniprot"].isin(uniprot_micro)

table_path = os.path.join(OUT_DIR, "microexon_phasesep_scores.csv")
merged.to_csv(table_path, index=False)
print(f"Saved: {table_path}")
print(f"Table rows: {len(merged)}, microexon: {merged['is_microexon'].sum()}")

Microexon genes: 166
PICNIC: 20449 rows, 168 microexon matches
PHPhunter: 20150 rows, 171 microexon matches

PICNIC:
  Microexon (n=168): median=0.672, mean=0.610
  All human (n=20281): median=0.413, mean=0.428
  Mann-Whitney U: U=2359481, p=7.48e-18
  Cliff's delta: 0.385

PHPhunter:
  Microexon (n=171): median=0.628, mean=0.584
  All human (n=19979): median=0.275, mean=0.331
  Mann-Whitney U: U=2633458, p=2.56e-34
  Cliff's delta: 0.542

Saved: ../outputs/microexon_phasesep_distribution.png
Saved: ../outputs/microexon_phasesep_distribution.svg
Saved: ../outputs/microexon_phasesep_scores.csv
Table rows: 20793, microexon: 168
